Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


Install independencies

In [2]:
!pip install transformers==4.37.2 --break-system-packages
!pip install accelerate==0.21.0 --break-system-packages

Load the model

In [3]:
import torch
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig

model_name = "OpenGVLab/InternVL2_5-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Loading InternVL2.5-8B...")
model = AutoModel.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    trust_remote_code=True,
    low_cpu_mem_usage=True
)
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model.eval()
print("InternVL2.5-8B loaded!")

Loading InternVL2.5-8B...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing fr

FlashAttention2 is not installed.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


...

./tokenizer.model:   0%|          | 0.00/1.48M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/179 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


InternVL2.5-8B loaded!


For one image

In [6]:
import torch
import torchvision.transforms as T
from PIL import Image
from torchvision.transforms.functional import InterpolationMode
import re

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def build_transform(input_size):
    transform = T.Compose([
        T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])
    return transform

def load_image(image_path, input_size=448):
    image = Image.open(image_path).convert('RGB')
    transform = build_transform(input_size)
    pixel_values = transform(image).unsqueeze(0)
    return pixel_values

image_path = "/content/drive/MyDrive/MyThesis2026/Chinese/Test/Test_images/31.jpg"
pixel_values = load_image(image_path).to(torch.float16).cuda()

prompt_text = """You are an expert in classifying harmful memes in the Chinese language. Your objective is to assess whether a meme is harmful or not.

Input: [Meme]
Image: [See attached image]
Text embedded: [Read from the image above]

Follow the steps below:
Step 1: Analyze the input meme by assessing the image and text to determine its harmfulness.
Step 2: If the meme contains any negative or insulting reference to gay or lesbian people, output Homophobia.
Step 3: If the meme contains any negative or insulting reference to transgender people, output Transphobia.
Step 4: If neither of the above applies, output Non_LGBT.

Output:
Your output should strictly follow the format:
Class labels: Homophobia, Transphobia, or Non_LGBT
Thought: Give your reason here"""

generation_config = dict(max_new_tokens=200, do_sample=False)

response = model.chat(tokenizer, pixel_values, prompt_text, generation_config)
print("RAW OUTPUT:", response)

m = re.search(r"(Homophobia|Transphobia|Non_LGBT)", response, re.IGNORECASE)
label = m.group(1) if m else None
print("PARSED LABEL:", label)

RAW OUTPUT: Class labels: Non_LGBT  
Thought: The meme uses the term "同志" which means "comrade" in Chinese, often used to refer to friends or allies. The image shows a person associated with this term, and the text "同志運動" translates to "Comrades' Movement." There is no negative or insulting reference to gay or lesbian people, nor is there any reference to transgender people. Therefore, the meme falls under the category of Non_LGBT.
PARSED LABEL: Non_LGBT


For multi images

删除错误结果

In [ ]:
import json

with open("/content/drive/MyDrive/Gemini25Flash_HM_ZeroShot_pred.json", "r", encoding="utf-8") as f:
    predictions = json.load(f)

print(f"当前已保存: {len(predictions)} 条")

当前已保存: 17 条


In [7]:
import os
import json
import re
import torch
import torchvision.transforms as T
from PIL import Image
from torchvision.transforms.functional import InterpolationMode
from tqdm import tqdm

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def build_transform(input_size):
    transform = T.Compose([
        T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])
    return transform

def load_image(image_path, input_size=448):
    image = Image.open(image_path).convert('RGB')
    transform = build_transform(input_size)
    pixel_values = transform(image).unsqueeze(0)
    return pixel_values

image_dir = "/content/drive/MyDrive/MyThesis2026/Chinese/Test/Test_images"
output_json = "/content/drive/MyDrive/InternVL25_8B_HM_ZeroShot_pred.json"

prompt_text = """You are an expert in classifying harmful memes in the Chinese language. Your objective is to assess whether a meme is harmful or not.

Input: [Meme]
Image: [See attached image]
Text embedded: [Read from the image above]

Follow the steps below:
Step 1: Analyze the input meme by assessing the image and text to determine its harmfulness.
Step 2: If the meme contains any negative or insulting reference to gay or lesbian people, output Homophobia.
Step 3: If the meme contains any negative or insulting reference to transgender people, output Transphobia.
Step 4: If neither of the above applies, output Non_LGBT.

Output:
Your output should strictly follow the format:
Class labels: Homophobia, Transphobia, or Non_LGBT
Thought: Give your reason here"""

generation_config = dict(max_new_tokens=200, do_sample=False)

image_files = sorted(
    [f for f in os.listdir(image_dir) if f.lower().endswith((".jpg", ".jpeg", ".png", ".gif"))],
    key=lambda x: int(re.search(r"(\d+)", x).group(1)) if re.search(r"(\d+)", x) else 0
)

# 断点续跑
if os.path.exists(output_json):
    with open(output_json, "r", encoding="utf-8") as f:
        predictions = json.load(f)
    predictions = [p for p in predictions if p["predicted_label"] != "ERROR"]
    done_images = {p["image_name"] for p in predictions}
    print(f"发现已有成功结果 {len(done_images)} 条，从断点继续...")
else:
    predictions = []
    done_images = set()
    print("没有已有结果，从头开始...")

remaining = [f for f in image_files if f not in done_images]
print(f"剩余待处理: {len(remaining)} 张")

for img_name in tqdm(remaining, desc="推理进度"):
    img_path = os.path.join(image_dir, img_name)

    try:
        torch.cuda.empty_cache()
        pixel_values = load_image(img_path).to(torch.float16).cuda()

        response = model.chat(tokenizer, pixel_values, prompt_text, generation_config)

        m = re.search(r"(Homophobia|Transphobia|Non_LGBT)", response, re.IGNORECASE)
        if m:
            label = m.group(1)
        elif any(w in response.lower() for w in ["unable", "cannot", "can't", "sorry"]):
            label = "Non_LGBT"
        else:
            label = "UNKNOWN"

        # 统一大小写
        if label.lower() == "homophobia":
            label = "Homophobia"
        elif label.lower() == "transphobia":
            label = "Transphobia"
        elif label.lower() == "non_lgbt":
            label = "Non_LGBT"

        predictions.append({
            "image_name": img_name,
            "predicted_label": label,
            "raw_output": response
        })

        print(f"✅ {img_name} -> {label}")

        with open(output_json, "w", encoding="utf-8") as f:
            json.dump(predictions, f, ensure_ascii=False, indent=2)

        torch.cuda.empty_cache()

    except Exception as e:
        print(f"❌ {img_name} 出错: {e}")
        predictions.append({
            "image_name": img_name,
            "predicted_label": "ERROR",
            "raw_output": str(e)
        })
        with open(output_json, "w", encoding="utf-8") as f:
            json.dump(predictions, f, ensure_ascii=False, indent=2)
        torch.cuda.empty_cache()

print(f"\n完成！共 {len(predictions)} 条结果已保存")
labels = [p["predicted_label"] for p in predictions]
for lbl in sorted(set(labels)):
    print(f"  {lbl}: {labels.count(lbl)}")

没有已有结果，从头开始...
剩余待处理: 232 张


推理进度:   0%|          | 1/232 [00:14<54:48, 14.23s/it]

✅ 1.jpg -> Non_LGBT


推理进度:   1%|          | 2/232 [00:26<50:44, 13.24s/it]

✅ 2.jpg -> Non_LGBT


...

✅ 237.jpg -> Non_LGBT


推理进度: 100%|█████████▉| 231/232 [37:59<00:10, 10.44s/it]

✅ 238.jpg -> Non_LGBT


推理进度: 100%|██████████| 232/232 [38:09<00:00,  9.87s/it]

✅ 239.jpg -> Non_LGBT

完成！共 232 条结果已保存
  Homophobia: 32
  Non_LGBT: 191
  Transphobia: 9


Calculate Zero-shot result

In [8]:
import json
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

json_path = "/content/drive/MyDrive/InternVL25_8B_HM_ZeroShot_pred.json"

with open(json_path, "r", encoding="utf-8") as f:
    predictions = json.load(f)

pred_df = pd.DataFrame(predictions)
pred_df["image_id"] = (
    pred_df["image_name"]
    .str.replace(".jpg", "", regex=False)
    .str.replace(".jpeg", "", regex=False)
    .str.replace(".png", "", regex=False)
    .str.replace(".gif", "", regex=False)
    .str.strip()
)

excel_path = "/content/drive/MyDrive/MyThesis2026/Chinese/Test/Test_labels.xlsx"
true_df = pd.read_excel(excel_path)
true_df.columns = true_df.columns.str.strip().str.lower()
true_df["image_id"] = true_df["id"].astype(str).str.strip()

merged_df = pd.merge(true_df, pred_df, on="image_id", how="inner")

print("Total True Labels:", len(true_df))
print("Total Predictions:", len(pred_df))
print("After Merge:", len(merged_df))

def normalize_label(x):
    x = str(x).strip().lower()
    if x in ["homophobia", "homophobic"]:
        return "homophobic"
    if x in ["transphobia", "transphobic"]:
        return "transphobic"
    if x in ["non_lgbt", "non-above", "non_anti_lgbt", "non anti lgbt"]:
        return "non anti lgbt"
    return x

merged_df["true_label"] = merged_df["label"].apply(normalize_label)
merged_df["predicted_label"] = merged_df["predicted_label"].apply(normalize_label)

print("\nUnique TRUE labels:", sorted(merged_df["true_label"].unique()))
print("Unique PRED labels:", sorted(merged_df["predicted_label"].unique()))

y_true = merged_df["true_label"]
y_pred = merged_df["predicted_label"]

ACC = accuracy_score(y_true, y_pred)
MP  = precision_score(y_true, y_pred, average="macro", zero_division=0)
MR  = recall_score(y_true, y_pred, average="macro", zero_division=0)
MF1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
WP  = precision_score(y_true, y_pred, average="weighted", zero_division=0)
WR  = recall_score(y_true, y_pred, average="weighted", zero_division=0)
WF1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print("\nEvaluation Metrics")
print("-------------------------------------------------")
print(f"ACC  : {ACC:.4f}")
print(f"MP   : {MP:.4f}")
print(f"MR   : {MR:.4f}")
print(f"MF1  : {MF1:.4f}")
print(f"WP   : {WP:.4f}")
print(f"WR   : {WR:.4f}")
print(f"WF1  : {WF1:.4f}")
print("-------------------------------------------------")

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

Total True Labels: 239
Total Predictions: 232
After Merge: 232

Unique TRUE labels: ['homophobic', 'non anti lgbt', 'transphobic']
Unique PRED labels: ['homophobic', 'non anti lgbt', 'transphobic']

Evaluation Metrics
-------------------------------------------------
ACC  : 0.3664
MP   : 0.5670
MR   : 0.4917
MF1  : 0.3582
WP   : 0.8095
WR   : 0.3664
WF1  : 0.3392
-------------------------------------------------

Confusion Matrix:
[[ 32 132   5]
 [  0  49   0]
 [  0  10   4]]

Classification Report:
               precision    recall  f1-score   support

   homophobic       1.00      0.19      0.32       169
non anti lgbt       0.26      1.00      0.41        49
  transphobic       0.44      0.29      0.35        14

     accuracy                           0.37       232
    macro avg       0.57      0.49      0.36       232
 weighted avg       0.81      0.37      0.34       232



Few-shot with multiple pics

In [9]:
import os
import json
import re
import torch
import numpy as np
import torchvision.transforms as T
from PIL import Image
from torchvision.transforms.functional import InterpolationMode
from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel

# ===============================
# 加载 CLIP + 训练集 embeddings
# ===============================
print("Loading CLIP...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

train_embeddings = np.load("/content/drive/MyDrive/train_embeddings.npy")
with open("/content/drive/MyDrive/train_meta.json", "r") as f:
    train_meta = json.load(f)

train_labels = train_meta["labels"]
train_filenames = train_meta["filenames"]
print(f"训练集 embeddings 加载完成，共 {len(train_embeddings)} 条")

# ===============================
# 路径配置
# ===============================
test_image_dir = "/content/drive/MyDrive/MyThesis2026/Chinese/Test/Test_images"
output_json = "/content/drive/MyDrive/InternVL25_8B_HM_FewShot_RAG_pred.json"

# ===============================
# 图片处理函数
# ===============================
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def build_transform(input_size):
    return T.Compose([
        T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])

def load_image(image_path, input_size=448):
    image = Image.open(image_path).convert('RGB')
    transform = build_transform(input_size)
    return transform(image).unsqueeze(0)

# ===============================
# RAG 检索函数
# ===============================
def get_clip_embedding(image_path):
    image = Image.open(image_path).convert("RGB")
    inputs = clip_processor(images=image, return_tensors="pt")
    with torch.no_grad():
        outputs = clip_model.vision_model(**inputs)
        emb = outputs.pooler_output
        emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb.squeeze().numpy()

def retrieve_examples(test_emb):
    similarities = train_embeddings @ test_emb
    examples = {}
    for label in ["Homophobic", "Transphobic", "Non_Anti_LGBT"]:
        label_indices = [i for i, l in enumerate(train_labels) if l == label]
        label_sims = [(i, similarities[i]) for i in label_indices]
        label_sims.sort(key=lambda x: x[1], reverse=True)
        examples[label] = label_sims[0][0]
    return examples

def build_prompt(example_indices):
    label_map = {
        "Homophobic": "Homophobia",
        "Transphobic": "Transphobia",
        "Non_Anti_LGBT": "Non_LGBT"
    }
    examples_text = ""
    for i, (label, idx) in enumerate(example_indices.items()):
        examples_text += f"Example {i+1}: Class label: {label_map[label]}\n"

    return f"""You are an expert in classifying harmful memes in the Chinese language. Your objective is to assess whether a meme is harmful or not.

Below are 3 reference examples with their correct labels retrieved from similar memes:

{examples_text}
Now classify the following meme:

Input: [Meme]
Image: [See attached image]
Text embedded: [Read from the image above]

Follow the steps below:
Step 1: Analyze the input meme by assessing the image and text, using the provided examples as reference to determine its harmfulness.
Step 2: If the meme contains any negative or insulting reference to gay or lesbian people, output Homophobia.
Step 3: If the meme contains any negative or insulting reference to transgender people, output Transphobia.
Step 4: If neither of the above applies, output Non_LGBT.

Output:
Your output should strictly follow the format:
Class labels: Homophobia, Transphobia, or Non_LGBT
Thought: Give your reason here"""

# ===============================
# 获取测试图片列表
# ===============================
image_files = sorted(
    [f for f in os.listdir(test_image_dir) if f.lower().endswith((".jpg", ".jpeg", ".png", ".gif"))],
    key=lambda x: int(re.search(r"(\d+)", x).group(1)) if re.search(r"(\d+)", x) else 0
)

# 断点续跑
if os.path.exists(output_json):
    with open(output_json, "r", encoding="utf-8") as f:
        predictions = json.load(f)
    predictions = [p for p in predictions if p["predicted_label"] != "ERROR"]
    done_images = {p["image_name"] for p in predictions}
    print(f"发现已有成功结果 {len(done_images)} 条，从断点继续...")
else:
    predictions = []
    done_images = set()
    print("没有已有结果，从头开始...")

remaining = [f for f in image_files if f not in done_images]
print(f"剩余待处理: {len(remaining)} 张")

generation_config = dict(max_new_tokens=200, do_sample=False)

# ===============================
# 批量推理
# ===============================
for img_name in tqdm(remaining, desc="推理进度"):
    img_path = os.path.join(test_image_dir, img_name)

    try:
        torch.cuda.empty_cache()

        # RAG 检索
        test_emb = get_clip_embedding(img_path)
        example_indices = retrieve_examples(test_emb)
        prompt_text = build_prompt(example_indices)

        # 加载图片
        pixel_values = load_image(img_path).to(torch.float16).cuda()

        # 推理
        response = model.chat(tokenizer, pixel_values, prompt_text, generation_config)

        m = re.search(r"(Homophobia|Transphobia|Non_LGBT)", response, re.IGNORECASE)
        if m:
            label = m.group(1)
        elif any(w in response.lower() for w in ["unable", "cannot", "can't", "sorry"]):
            label = "Non_LGBT"
        else:
            label = "UNKNOWN"

        # 统一大小写
        if label.lower() == "homophobia":
            label = "Homophobia"
        elif label.lower() == "transphobia":
            label = "Transphobia"
        elif label.lower() == "non_lgbt":
            label = "Non_LGBT"

        predictions.append({
            "image_name": img_name,
            "predicted_label": label,
            "raw_output": response
        })

        print(f"✅ {img_name} -> {label}")

        with open(output_json, "w", encoding="utf-8") as f:
            json.dump(predictions, f, ensure_ascii=False, indent=2)

        torch.cuda.empty_cache()

    except Exception as e:
        print(f"❌ {img_name} 出错: {e}")
        predictions.append({
            "image_name": img_name,
            "predicted_label": "ERROR",
            "raw_output": str(e)
        })
        with open(output_json, "w", encoding="utf-8") as f:
            json.dump(predictions, f, ensure_ascii=False, indent=2)
        torch.cuda.empty_cache()

print(f"\n完成！共 {len(predictions)} 条结果已保存")
labels = [p["predicted_label"] for p in predictions]
for lbl in sorted(set(labels)):
    print(f"  {lbl}: {labels.count(lbl)}")

Loading CLIP...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

...

✅ 237.jpg -> Non_LGBT


推理进度: 100%|█████████▉| 231/232 [41:35<00:11, 11.91s/it]

✅ 238.jpg -> Non_LGBT


推理进度: 100%|██████████| 232/232 [41:47<00:00, 10.81s/it]

✅ 239.jpg -> Non_LGBT

完成！共 232 条结果已保存
  Homophobia: 35
  Non_LGBT: 193
  Transphobia: 4


 Calculation of Few shot

In [10]:
import json
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

json_path = "/content/drive/MyDrive/InternVL25_8B_HM_FewShot_RAG_pred.json"

with open(json_path, "r", encoding="utf-8") as f:
    predictions = json.load(f)

pred_df = pd.DataFrame(predictions)
pred_df["image_id"] = (
    pred_df["image_name"]
    .str.replace(".jpg", "", regex=False)
    .str.replace(".jpeg", "", regex=False)
    .str.replace(".png", "", regex=False)
    .str.replace(".gif", "", regex=False)
    .str.strip()
)

excel_path = "/content/drive/MyDrive/MyThesis2026/Chinese/Test/Test_labels.xlsx"
true_df = pd.read_excel(excel_path)
true_df.columns = true_df.columns.str.strip().str.lower()
true_df["image_id"] = true_df["id"].astype(str).str.strip()

merged_df = pd.merge(true_df, pred_df, on="image_id", how="inner")

print("Total True Labels:", len(true_df))
print("Total Predictions:", len(pred_df))
print("After Merge:", len(merged_df))

def normalize_label(x):
    x = str(x).strip().lower()
    if x in ["homophobia", "homophobic"]:
        return "homophobic"
    if x in ["transphobia", "transphobic"]:
        return "transphobic"
    if x in ["non_lgbt", "non-above", "non_anti_lgbt", "non anti lgbt"]:
        return "non anti lgbt"
    return x

merged_df["true_label"] = merged_df["label"].apply(normalize_label)
merged_df["predicted_label"] = merged_df["predicted_label"].apply(normalize_label)

print("\nUnique TRUE labels:", sorted(merged_df["true_label"].unique()))
print("Unique PRED labels:", sorted(merged_df["predicted_label"].unique()))

y_true = merged_df["true_label"]
y_pred = merged_df["predicted_label"]

ACC = accuracy_score(y_true, y_pred)
MP  = precision_score(y_true, y_pred, average="macro", zero_division=0)
MR  = recall_score(y_true, y_pred, average="macro", zero_division=0)
MF1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
WP  = precision_score(y_true, y_pred, average="weighted", zero_division=0)
WR  = recall_score(y_true, y_pred, average="weighted", zero_division=0)
WF1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print("\nEvaluation Metrics")
print("-------------------------------------------------")
print(f"ACC  : {ACC:.4f}")
print(f"MP   : {MP:.4f}")
print(f"MR   : {MR:.4f}")
print(f"MF1  : {MF1:.4f}")
print(f"WP   : {WP:.4f}")
print(f"WR   : {WR:.4f}")
print(f"WF1  : {WF1:.4f}")
print("-------------------------------------------------")

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

Total True Labels: 239
Total Predictions: 232
After Merge: 232

Unique TRUE labels: ['homophobic', 'non anti lgbt', 'transphobic']
Unique PRED labels: ['homophobic', 'non anti lgbt', 'transphobic']

Evaluation Metrics
-------------------------------------------------
ACC  : 0.3793
MP   : 0.7513
MR   : 0.4976
MF1  : 0.3975
WP   : 0.8424
WR   : 0.3793
WF1  : 0.3623
-------------------------------------------------

Confusion Matrix:
[[ 35 134   0]
 [  0  49   0]
 [  0  10   4]]

Classification Report:
               precision    recall  f1-score   support

   homophobic       1.00      0.21      0.34       169
non anti lgbt       0.25      1.00      0.40        49
  transphobic       1.00      0.29      0.44        14

     accuracy                           0.38       232
    macro avg       0.75      0.50      0.40       232
 weighted avg       0.84      0.38      0.36       232

